In [ ]:
# Copyright © 2025 Haocheng Wen, Faxuan Luo, Hanbing Zou
# SPDX-License-Identifier: MIT

!pip install git+https://github.com/JA4S/JAX-AMR.git
!wget https://raw.githubusercontent.com/JA4S/JAX-AMR/main/examples/simple_solver.py

In [ ]:

from jaxamr import amr, amraux, simple_solver

import jax
import jax.numpy as jnp
from jaxamr.amr import remove_ghost,recover_ghost
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from jaxamr.simple_solver import rk2_L0,rk2
import numpy as np

jax.config.update("jax_enable_x64", False)
jax.config.update('jax_platform_name', 'cpu')

amr基础设置
AMR Basic Settings

In [ ]:
Lx = 1.0
Ly = 1.0

nx = 80
ny = 80

dx = Lx/nx
dy = Ly/ny

grid_config = {'dx':dx,'dy':dy}
base_grid = {'Lx':Lx,'Ly':Ly,'Nx':nx,'Ny':ny}

n_block = [
    [1, 1],  # Level 0
    [4, 4], # Level 1
    [2, 2],  # Level 2
    [2, 2],  # Level 3
    [2, 2]   # Level 4
    ] # x-direction, y-direction

template_node_num = 3
buffer_num = 5
refinement_tolerance = {
    'density': 0.1,
}
amr_config = {'base_grid':base_grid,
        'n_block':n_block,
        'template_node_num':template_node_num,
        'buffer_num':buffer_num,
        'refinement_tolerance':refinement_tolerance
}
amr.set_amr(amr_config)
dx = [dx]
dy = [dy]
for i, (bx, by) in enumerate(n_block[1:], 1):
    dx.append(Lx/nx / (2.0**i))
    dy.append(Ly/ny / (2.0**i))

初始化
Initilization

In [ ]:
def initial_conditions(nx, ny, gamma=1.4):
    x = jnp.linspace(0, 1, nx)
    y = jnp.linspace(0, 1, ny)
    X, Y = jnp.meshgrid(x, y, indexing='ij')

    rho = jnp.where(jnp.sqrt((X - 0.5)**2 + (Y - 0.5)**2) < 0.15, 1.0, 0.125)
    u = jnp.zeros_like(X)
    v = jnp.zeros_like(X)
    p = jnp.where(jnp.sqrt((X - 0.5)**2 + (Y - 0.5)**2) < 0.15, 1.0, 0.1)
    E = p / (gamma - 1) + 0.5 * rho * (u**2 + v**2)

    U = jnp.array([rho, rho * u, rho * v, E])
    return X, Y, U

_,_,U = initial_conditions(nx,ny)

blk_data0 = jnp.array([U])

blk_info0 = {
      'number': 1,
      'index': jnp.array([0, 0]),
      'glob_index': jnp.array([[0, 0]]),
      'neighbor_index': jnp.array([[-1, -1, -1, -1]])
        }
theta = {}

AMR迭代
AMR main loop

In [ ]:
nt = 2001 # computation steps
amr_update_step = 5 # AMR update steps
time = 0
amr_initialized = False
for step in tqdm(range(nt), desc="Progress", unit="step"):
    dt = 2e-5
    time += dt
    if amr_initialized == False :

        blk_data1, blk_info1, max_blk_num1 = amr.initialize(1, blk_data0, blk_info0, dx[1], dy[1])
        blk_data2, blk_info2, max_blk_num2 = amr.initialize(2, blk_data1, blk_info1, dx[2], dy[2])
        blk_data3, blk_info3, max_blk_num3 = amr.initialize(3, blk_data2, blk_info2, dx[3], dy[3])

        amr_initialized = True

    elif (step % amr_update_step == 0):
        blk_data1, blk_info1, max_blk_num1 = amr.update(1, blk_data0, blk_info0, dx[1], dy[1], blk_data1, blk_info1, max_blk_num1)
        blk_data2, blk_info2, max_blk_num2 = amr.update(2, blk_data1, blk_info1, dx[2], dy[2], blk_data2, blk_info2, max_blk_num2)
        blk_data3, blk_info3, max_blk_num3 = amr.update(3, blk_data2, blk_info2, dx[3], dy[3], blk_data3, blk_info3, max_blk_num3)


    pre_blk_data0 = blk_data0
    pre_blk_data1 = recover_ghost(1, pre_blk_data0, blk_data1, blk_info1, theta=theta)
    pre_blk_data2 = recover_ghost(2, pre_blk_data1, blk_data2, blk_info2, theta=theta)
    pre_blk_data3 = recover_ghost(3, pre_blk_data2, blk_data3, blk_info3, theta=theta)
    val_num1 = int(np.ceil(blk_info1['number'] / 100) * 100)
    val_num2 = int(np.ceil(blk_info2['number'] / 100) * 100)
    val_num3 = int(np.ceil(blk_info3['number'] / 100) * 100)
    act_blk_data1_with_ghost = pre_blk_data1[:val_num1,:,:,:]
    act_blk_data2_with_ghost = pre_blk_data2[:val_num2,:,:,:]
    act_blk_data3_with_ghost = pre_blk_data3[:val_num3,:,:,:]
    inact_blk_data1_with_ghost = remove_ghost(pre_blk_data1[val_num1:,:,:,:])
    inact_blk_data2_with_ghost = remove_ghost(pre_blk_data2[val_num2:,:,:,:])
    inact_blk_data3_with_ghost = remove_ghost(pre_blk_data3[val_num3:,:,:,:])


    blk_data0_new = rk2_L0(pre_blk_data0, dx[0], dy[0], dt/8.0,theta)
    blk_data1_new = rk2(act_blk_data1_with_ghost, dx[1], dy[1], dt/8.0,theta)
    blk_data2_new = rk2(act_blk_data2_with_ghost, dx[2], dy[2], dt/8.0,theta)    
    blk_data3_new = rk2(act_blk_data3_with_ghost, dx[3], dy[3], dt/8.0,theta)    

    blk_data1_new = remove_ghost(blk_data1_new)
    blk_data2_new = remove_ghost(blk_data2_new)
    blk_data3_new = remove_ghost(blk_data3_new)
    blk_data1_new = jnp.concatenate((blk_data1_new,inact_blk_data1_with_ghost),axis=0)
    blk_data2_new = jnp.concatenate((blk_data2_new,inact_blk_data2_with_ghost),axis=0)
    blk_data3_new = jnp.concatenate((blk_data3_new,inact_blk_data3_with_ghost),axis=0)

    blk_data2 = amr.interpolate_fine_to_coarse(3, blk_data2_new, blk_data3_new, blk_info3)
    blk_data1 = amr.interpolate_fine_to_coarse(2, blk_data1_new, blk_data2_new, blk_info2)
    blk_data0 = amr.interpolate_fine_to_coarse(1, blk_data0_new, blk_data1_new, blk_info1)

    blk_data3 = blk_data3_new


后处理
Result Visualization

In [ ]:
# Density Contour
plt.figure(figsize=(10, 10))
ax = plt.gca()

component = 0
vrange = (0, 1)
fig = amraux.plot_block_data(blk_data0[:, component], blk_info0, ax, vrange) # Level 0
fig = amraux.plot_block_data(blk_data1[:, component], blk_info1, ax, vrange) # Level 1
fig = amraux.plot_block_data(blk_data2[:, component], blk_info2, ax, vrange) # Level 2
fig = amraux.plot_block_data(blk_data3[:, component], blk_info3, ax, vrange) # Level 3

plt.colorbar(fig, ax=ax, label='Density')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.axis('equal')
plt.show()

# Refinement Level
plt.figure(figsize=(10, 8))
ax = plt.gca()

component = 0
vrange = (0, 3)
fig = amraux.plot_block_data(0*jnp.ones_like(blk_data0[:, component]), blk_info0, ax, vrange) # Level 0
fig = amraux.plot_block_data(1*jnp.ones_like(blk_data1[:, component]), blk_info1, ax, vrange) # Level 1
fig = amraux.plot_block_data(2*jnp.ones_like(blk_data2[:, component]), blk_info2, ax, vrange) # Level 2
fig = amraux.plot_block_data(3*jnp.ones_like(blk_data3[:, component]), blk_info3, ax, vrange) # Level 3

plt.colorbar(fig, ax=ax, label='Refinement Level')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.axis('equal')
plt.show()